# BTCUSDT Price Prediction - Complete Example

## Setup

Install once from the repo root (use `.venv` as the notebook kernel):

```bash
pip install -e ".[prediction]"
```

Imports use the `src` package (e.g. `from src.data.data_load import load_data`).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import xgboost
import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

## Step 1: Load Data

In [3]:
from src.data.data_load import load_data

# Load BTCUSDT 1-minute kline data
df = load_data(
    symbols=['BTCUSDT'],
    start_date="2025-01-01",
    end_date="2025-01-31",
    interval="1m"
)

print(f"Data shape: {df.shape}")
print(f"\nData info:")
print(df.info())
print(f"\nFirst few rows:")
print(df.head())

Data shape: (43201, 13)

Data info:
<class 'pandas.DataFrame'>
RangeIndex: 43201 entries, 0 to 43200
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   open_time        43201 non-null  datetime64[ms, UTC]
 1   open             43201 non-null  float64            
 2   high             43201 non-null  float64            
 3   low              43201 non-null  float64            
 4   close            43201 non-null  float64            
 5   volume           43201 non-null  float64            
 6   quote_volume     43201 non-null  float64            
 7   num_trades       43201 non-null  int64              
 8   taker_base_vol   43201 non-null  float64            
 9   taker_quote_vol  43201 non-null  float64            
 10  close_time       43201 non-null  int64              
 11  symbol           43201 non-null  str                
 12  year             43201 non-null  int32           

In [4]:
# Data statistics
print("Data statistics:")
print(df[['open', 'high', 'low', 'close', 'volume']].describe())

Data statistics:
                open           high            low          close  \
count   43201.000000   43201.000000   43201.000000   43201.000000   
mean    99853.459043   99892.972985   99814.005142   99853.716523   
std      4288.987795    4294.296069    4283.148048    4288.950099   
min     89417.870000   90172.490000   89256.690000   89417.880000   
25%     95963.700000   96000.000000   95921.710000   95964.360000   
50%     99782.700000   99819.120000   99746.000000   99782.700000   
75%    104043.990000  104098.250000  103989.790000  104044.800000   
max    109185.870000  109588.000000  108945.080000  109194.170000   

             volume  
count  43201.000000  
mean      19.503367  
std       34.798929  
min        0.193000  
25%        4.821630  
50%       10.063310  
75%       21.084190  
max     1417.439912  


## Step 2: Feature Engineering

In [5]:
from src.strategy.predictive import PriceFeatures, prepare_data_for_modeling

# Prepare features and target variable
print("Generating features...")
df_features, feature_cols, target_col = prepare_data_for_modeling(
    df,
    horizon=1,  # Predict 1 minute ahead
    lookback_window=60,
    dropna=True
)

print(f"\nDataset shape after feature engineering: {df_features.shape}")
print(f"Number of features: {len(feature_cols)}")
print(f"\nFeature columns: {feature_cols[:10]}...")  # Show first 10 features
print(f"\nTarget distribution:")
print(df_features[target_col].value_counts())
print(f"Target label balance: {df_features[target_col].mean():.2%} positive class")

Generating features...

Dataset shape after feature engineering: (43101, 76)
Number of features: 64

Feature columns: ['quote_volume', 'num_trades', 'taker_base_vol', 'taker_quote_vol', 'year', 'return', 'log_return', 'hl_ratio', 'co_ratio', 'cc_ratio']...

Target distribution:
target_binary
0    22001
1    21100
Name: count, dtype: int64
Target label balance: 48.95% positive class


In [6]:
# Display all feature columns
print(f"All features ({len(feature_cols)} total):")
for i, col in enumerate(feature_cols, 1):
    print(f"{i:3d}. {col}")

All features (64 total):
  1. quote_volume
  2. num_trades
  3. taker_base_vol
  4. taker_quote_vol
  5. year
  6. return
  7. log_return
  8. hl_ratio
  9. co_ratio
 10. cc_ratio
 11. ho_ratio
 12. lo_ratio
 13. sma_5
 14. sma_10
 15. sma_20
 16. sma_50
 17. sma_100
 18. ema_5
 19. ema_10
 20. ema_20
 21. ema_50
 22. ema_100
 23. momentum_5
 24. momentum_10
 25. momentum_20
 26. momentum_60
 27. rsi_14
 28. rsi_7
 29. macd
 30. macd_signal
 31. macd_hist
 32. bb_upper
 33. bb_middle
 34. bb_lower
 35. bb_width
 36. bb_position
 37. tr
 38. atr
 39. volume_sma_5
 40. volume_sma_10
 41. volume_sma_20
 42. volume_ratio
 43. obv
 44. obv_sma
 45. volatility_5
 46. volatility_10
 47. volatility_20
 48. volatility_60
 49. price_volume_corr
 50. close_lag_1
 51. return_lag_1
 52. volume_lag_1
 53. close_lag_2
 54. return_lag_2
 55. volume_lag_2
 56. close_lag_3
 57. return_lag_3
 58. volume_lag_3
 59. close_lag_4
 60. return_lag_4
 61. volume_lag_4
 62. close_lag_5
 63. return_lag_5
 64. vol

## Step 3: Data Preparation

In [7]:
from src.strategy.predictive import TimeSeriesSplitter

# Option 1: Split by date ranges
train_start = "2025-01-01"
train_end = "2025-01-20"
test_start = "2025-01-21"
test_end = "2025-01-31"

X_train, X_val, X_test, y_train, y_val, y_test = TimeSeriesSplitter.train_test_split_by_date(
    df_features,
    feature_cols,
    target_col,
    train_start_date=train_start,
    train_end_date=train_end,
    test_start_date=test_start,
    test_end_date=test_end,
    val_ratio=0.1  # 10% of training data for validation
)

# Option 2: Split by periods (days)
# X_train, X_val, X_test, y_train, y_val, y_test = TimeSeriesSplitter.train_test_split_by_period(
#     df_features,
#     feature_cols,
#     target_col,
#     train_period_days=365,  # 1 year training
#     test_period_days=90,    # 3 months testing
#     val_ratio=0.1
# )

Date-based time series split:
  Train: 24536 samples (2025-01-01 to 2025-01-20, 10.0% for validation)
  Val:   2726 samples (from training data)
  Test:  14400 samples (2025-01-21 to 2025-01-31)
  Total: 41662 samples



## Step 4: Train Models

In [8]:

import xgboost

In [9]:
from src.strategy.predictive import (
    LinearModel,
    RandomForestModel,
    GradientBoostingModel,
    XGBoostModel,
    MLPModel,
    ModelTrainer
)

# Initialize trainer
trainer = ModelTrainer()

# Initialize models
models = [
    LinearModel(),
    RandomForestModel(n_estimators=100, max_depth=15),
    XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1),
]

# Train traditional ML models
for model in models:
    trainer.train_model(
        model,
        X_train, y_train,
        X_val, y_val
    )

Training Linear model...
Linear Model trained
Linear model trained successfully!

Training RandomForest model...
RandomForest Model trained
RandomForest model trained successfully!

Training XGBoost model...
XGBoost Model trained
XGBoost model trained successfully!



In [10]:
# Train MLP model (simple neural network)
mlp_model = MLPModel(hidden_layers=[128, 64, 32], epochs=20, batch_size=32)
trainer.train_model(
    mlp_model,
    X_train, y_train,
    X_val, y_val,
    verbose=1
)

Training MLP model...
Epoch 10/20, Loss: 0.6813
Epoch 20/20, Loss: 0.6789
MLP Model trained
MLP model trained successfully!



In [11]:
try:
    from src.strategy.predictive import CNNModel

    print("Training CNN model...")
    cnn_model = CNNModel(epochs=30, batch_size=32)

    cnn_model.train(
        X_train, y_train,
        seq_length=20,  # Use 20-minute lookback
        X_val=X_val, y_val=y_val,
        verbose=1
    )

    # Evaluate CNN
    trainer.models['CNN'] = cnn_model
    trainer.evaluate_model(cnn_model, X_test, y_test)

except ImportError as e:
    print(f"PyTorch not installed: {e}")
    print("Install with: pip install torch")

Training CNN model...
Epoch 10/30, Loss: 0.6943
Epoch 20/30, Loss: 0.2700
Epoch 30/30, Loss: 0.3005
CNN Model trained
Evaluating CNN model...

CNN - Evaluation Results
Accuracy:  0.5140
Precision: 0.5086
Recall:    0.4800
F1-Score:  0.4939

Confusion Matrix:
[[3988 3299]
 [3699 3414]]



## Step 5: Evaluate Models on Test Set

In [12]:
# Evaluate all trained models
for model_name, model in trainer.models.items():
    trainer.evaluate_model(model, X_test, y_test)

Evaluating Linear model...

Linear - Evaluation Results
Accuracy:  0.5238
Precision: 0.5263
Recall:    0.3607
F1-Score:  0.4281
AUC-ROC:   0.5258

Confusion Matrix:
[[4977 2310]
 [4547 2566]]

Evaluating RandomForest model...

RandomForest - Evaluation Results
Accuracy:  0.5128
Precision: 0.5112
Recall:    0.3141
F1-Score:  0.3891
AUC-ROC:   0.5168

Confusion Matrix:
[[5151 2136]
 [4879 2234]]

Evaluating XGBoost model...

XGBoost - Evaluation Results
Accuracy:  0.5136
Precision: 0.5115
Recall:    0.3395
F1-Score:  0.4081
AUC-ROC:   0.5241

Confusion Matrix:
[[4981 2306]
 [4698 2415]]

Evaluating MLP model...

MLP - Evaluation Results
Accuracy:  0.5111
Precision: 0.5449
Recall:    0.0623
F1-Score:  0.1118
AUC-ROC:   0.5280

Confusion Matrix:
[[6917  370]
 [6670  443]]

Evaluating CNN model...

CNN - Evaluation Results
Accuracy:  0.5140
Precision: 0.5086
Recall:    0.4800
F1-Score:  0.4939

Confusion Matrix:
[[3988 3299]
 [3699 3414]]



## Step 6: Model Comparison

In [13]:
# Compare all models
df_comparison = trainer.compare_models()


MODEL COMPARISON
       Model  Accuracy  Precision   Recall  F1-Score      AUC
         CNN  0.514028   0.508565 0.479966  0.493852      NaN
      Linear  0.523819   0.526251 0.360748  0.428059 0.525837
     XGBoost  0.513611   0.511544 0.339519  0.408146 0.524071
RandomForest  0.512847   0.511213 0.314073  0.389097 0.516759
         MLP  0.511111   0.544895 0.062280  0.111784 0.528050



## Step 7: Feature Importance (for tree-based models)

In [14]:
# Get feature importance from tree-based models
if 'RandomForest' in trainer.models:
    rf_model = trainer.models['RandomForest']
    rf_model.feature_cols = feature_cols
    
    importance_df = rf_model.feature_importance()
    print(f"\nRandom Forest - Top 20 Important Features:")
    print(importance_df.head(20))


Random Forest - Top 20 Important Features:
              feature  importance
50       return_lag_1    0.021846
56       return_lag_3    0.021842
53       return_lag_2    0.021472
63       volume_lag_5    0.021290
62       return_lag_5    0.021239
59       return_lag_4    0.021202
60       volume_lag_4    0.020958
54       volume_lag_2    0.020697
47      volatility_60    0.020545
34           bb_width    0.020540
48  price_volume_corr    0.020035
57       volume_lag_3    0.019903
51       volume_lag_1    0.019846
26             rsi_14    0.019772
44       volatility_5    0.019700
25        momentum_60    0.019513
23        momentum_10    0.019429
41       volume_ratio    0.019284
27              rsi_7    0.019031
30          macd_hist    0.019030


## Summary

### Popular Models for Price Prediction:

1. **Linear Models**
   - Logistic Regression: Simple, fast, interpretable baseline
   - Best for: Quick prototyping, understanding feature importance

2. **Tree-based Models**
   - Random Forest: Robust, handles non-linearity, resistant to overfitting
   - XGBoost: High accuracy, efficient, handles missing values well
   - Gradient Boosting: Strong ensemble method, good for structured data
   - Best for: High-dimensional feature data, good generalization

3. **Deep Learning**
   - MLP: Simple neural network, good for tabular data
   - LSTM: Captures long-term dependencies in sequences
   - CNN: Good for local patterns in sequential data
   - Transformer: State-of-the-art for sequences, captures complex patterns
   - Best for: Raw sequential data, learning temporal patterns

### Key Features for Price Prediction:

1. **Technical Indicators**: RSI, MACD, Bollinger Bands, ATR
2. **Moving Averages**: SMA, EMA of different periods
3. **Momentum**: Price momentum over different lookback periods
4. **Volume**: Volume-weighted indicators, OBV
5. **Volatility**: Standard deviation of returns
6. **Lagged Features**: Previous candle prices and returns
7. **Price Ratios**: HL ratio, CO ratio, etc.

### Recommendations:

- Start with **tree-based models** (Random Forest/XGBoost) for good balance of accuracy and speed
- Use **LSTM** or **Transformer** if you have enough data and computational resources
- Combine predictions from multiple models using ensemble methods for best results
- Always validate on out-of-sample test data with proper time series split
- Monitor for concept drift in financial models

## Additional Notes:

- **LSTM Model Training**: If using the LSTM model, ensure PyTorch is installed. The LSTM model captures long-term dependencies in the price sequence, potentially improving prediction performance.
- **Hyperparameter Tuning**: Consider tuning model hyperparameters (e.g., learning rate, batch size, number of layers) for optimal performance.
- **Ensemble Methods**: Combining predictions from multiple models (e.g., averaging, stacking) can lead to more robust and accurate predictions.
- **Feature Selection**: Regularly review and select important features to reduce model complexity and improve interpretability.
- **Model Retraining**: Periodically retrain models with new data to maintain prediction accuracy over time.

### LSTM Model

In [15]:
try:
    from src.strategy.predictive import LSTMModel

    print("Training LSTM model...")
    lstm_model = LSTMModel(epochs=30, batch_size=32, lstm_units=50)

    lstm_model.train(
        X_train, y_train,
        seq_length=60,  # Use 60-minute lookback
        X_val=X_val, y_val=y_val,
        verbose=1
    )

    # Evaluate LSTM
    trainer.models['LSTM'] = lstm_model
    trainer.evaluate_model(lstm_model, X_test, y_test)

except ImportError as e:
    print(f"PyTorch not installed: {e}")
    print("Install with: pip install torch")

Training LSTM model...
Epoch 10/30, Loss: 0.6908
Epoch 20/30, Loss: 0.6883
Epoch 30/30, Loss: 0.6872
LSTM Model trained
Evaluating LSTM model...

LSTM - Evaluation Results
Accuracy:  0.5103
Precision: 0.5047
Recall:    0.4639
F1-Score:  0.4834

Confusion Matrix:
[[4048 3239]
 [3813 3300]]



### Transformer Model

In [16]:
try:
    from src.strategy.predictive import TransformerModel

    print("Training Transformer model...")
    transformer_model = TransformerModel(epochs=30, batch_size=32, d_model=64, num_heads=4)

    transformer_model.train(
        X_train, y_train,
        seq_length=60,  # Use 60-minute lookback
        X_val=X_val, y_val=y_val,
        verbose=1
    )

    # Evaluate Transformer
    trainer.models['Transformer'] = transformer_model
    trainer.evaluate_model(transformer_model, X_test, y_test)

except ImportError as e:
    print(f"PyTorch not installed: {e}")
    print("Install with: pip install torch")


Training Transformer model...
Epoch 10/30, Loss: 0.6946
Epoch 20/30, Loss: 0.6925
Epoch 30/30, Loss: 0.6932
Transformer Model trained
Evaluating Transformer model...

Transformer - Evaluation Results
Accuracy:  0.5122
Precision: 0.5047
Recall:    0.6737
F1-Score:  0.5771

Confusion Matrix:
[[2584 4703]
 [2321 4792]]

